# Semana 3: Modelamiento Conceptual y Lógico
## Notebook de Ejercicios (NB2) – Implementación en SQL Fiddle

### Objetivo
Llevar a la práctica los modelos lógicos diseñados en el NB1, implementándolos en un motor SQL real utilizando **DB-Fiddle**. Crearemos tablas, definiremos constraints (claves primarias, foráneas, checks) y probaremos la integridad de los datos mediante inserciones.

### Herramienta
Utilizaremos [DB-Fiddle](https://www.db-fiddle.com/), un editor SQL online gratuito que soporta PostgreSQL, MySQL, SQLite y más.

### Instrucciones generales
1.  Abre [DB-Fiddle](https://www.db-fiddle.com/) en tu navegador.
2.  Selecciona el motor **PostgreSQL** (o MySQL, según prefieras).
3.  En el panel superior (esquema), escribirás las sentencias `CREATE TABLE`.
4.  En el panel inferior (consultas), escribirás las inserciones y pruebas.
5.  Ejecuta con el botón "Run".
6.  Copia los resultados y reflexiona sobre ellos.

---
## Actividad 1: Implementación del modelo de Biblioteca

Recordemos el modelo lógico de la biblioteca que diseñamos en el NB1:

```sql
Tablas:
- Autor (id_autor, nombre, nacionalidad, fecha_nac)
- Libro (isbn, titulo, anio, editorial)
- Socio (id_socio, nombre, direccion, telefono, email)
- Prestamo (id_prestamo, fecha_prestamo, fecha_devolucion, id_socio, isbn)
- Libro_Autor (id_autor, isbn)
```

### 1.1. Crear las tablas con constraints

Abre DB-Fiddle y en el panel **Schema** (panel superior), pega el siguiente código SQL. Luego haz clic en **Run**.

In [ ]:
# Código SQL para crear las tablas (cópialo y pégalo en DB-Fiddle)
sql_biblioteca = """
-- Tabla Autor
CREATE TABLE autor (
    id_autor SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    nacionalidad VARCHAR(50),
    fecha_nac DATE
);

-- Tabla Libro
CREATE TABLE libro (
    isbn VARCHAR(20) PRIMARY KEY,
    titulo VARCHAR(200) NOT NULL,
    anio INT CHECK (anio > 0),
    editorial VARCHAR(100)
);

-- Tabla Socio
CREATE TABLE socio (
    id_socio SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    direccion VARCHAR(200),
    telefono VARCHAR(20),
    email VARCHAR(100) UNIQUE
);

-- Tabla Prestamo
CREATE TABLE prestamo (
    id_prestamo SERIAL PRIMARY KEY,
    fecha_prestamo DATE NOT NULL DEFAULT CURRENT_DATE,
    fecha_devolucion DATE,
    id_socio INT NOT NULL REFERENCES socio(id_socio),
    isbn VARCHAR(20) NOT NULL REFERENCES libro(isbn)
);

-- Tabla intermedia Libro_Autor (relación N:M)
CREATE TABLE libro_autor (
    id_autor INT REFERENCES autor(id_autor),
    isbn VARCHAR(20) REFERENCES libro(isbn),
    PRIMARY KEY (id_autor, isbn)
);
"""
print("✅ Código SQL para biblioteca listo. Cópialo y pégalo en DB-Fiddle.")

### 1.2. Insertar datos de prueba

Ahora, en el panel **Query** (panel inferior) de DB-Fiddle, pega y ejecuta las siguientes inserciones:

In [ ]:
# Código SQL para insertar datos de prueba (pégarlo en DB-Fiddle)
inserts_biblioteca = """
-- Insertar autores
INSERT INTO autor (nombre, nacionalidad, fecha_nac) VALUES
    ('Gabriel García Márquez', 'Colombiana', '1927-03-06'),
    ('Isabel Allende', 'Chilena', '1942-08-02'),
    ('Julio Cortázar', 'Argentina', '1914-08-26');

-- Insertar libros
INSERT INTO libro (isbn, titulo, anio, editorial) VALUES
    ('978-84-9759-382-8', 'Cien años de soledad', 1967, 'Sudamericana'),
    ('978-84-9759-383-5', 'El amor en los tiempos del cólera', 1985, 'Sudamericana'),
    ('978-84-9759-384-2', 'La casa de los espíritus', 1982, 'Plaza & Janés'),
    ('978-84-9759-385-9', 'Rayuela', 1963, 'Sudamericana');

-- Relacionar autores con libros (libro_autor)
INSERT INTO libro_autor (id_autor, isbn) VALUES
    (1, '978-84-9759-382-8'), -- García Márquez escribe Cien años...
    (1, '978-84-9759-383-5'), -- García Márquez escribe El amor...
    (2, '978-84-9759-384-2'), -- Isabel Allende escribe La casa...
    (3, '978-84-9759-385-9'); -- Cortázar escribe Rayuela

-- Insertar socios
INSERT INTO socio (nombre, direccion, telefono, email) VALUES
    ('Juan Pérez', 'Calle Falsa 123', '555-1234', 'juan@email.com'),
    ('María García', 'Avenida Siempreviva 456', '555-5678', 'maria@email.com');

-- Insertar préstamos
INSERT INTO prestamo (fecha_prestamo, fecha_devolucion, id_socio, isbn) VALUES
    ('2025-03-01', '2025-03-15', 1, '978-84-9759-382-8'),
    ('2025-03-05', NULL, 2, '978-84-9759-384-2');

-- Consultas de verificación
SELECT * FROM autor;
SELECT * FROM libro;
SELECT * FROM socio;
SELECT * FROM prestamo;
"""
print("✅ Código de inserciones listo. Cópialo y pruébalo en DB-Fiddle.")

### 1.3. Probar la integridad referencial

Intenta ejecutar las siguientes sentencias que **deberían fallar** debido a las constraints. Esto demuestra que la integridad está funcionando.

**Inserta en DB-Fiddle y observa el error:**

In [ ]:
# Código para probar violaciones de integridad
violaciones = """
-- 1. Insertar un préstamo con un socio que no existe (debe fallar)
INSERT INTO prestamo (fecha_prestamo, id_socio, isbn) VALUES ('2025-03-10', 999, '978-84-9759-382-8');

-- 2. Insertar un préstamo con un ISBN de libro que no existe (debe fallar)
INSERT INTO prestamo (fecha_prestamo, id_socio, isbn) VALUES ('2025-03-10', 1, '000-00-0000-000-0');

-- 3. Insertar un libro con año negativo (viola CHECK)
INSERT INTO libro (isbn, titulo, anio) VALUES ('000-00-0000-000-0', 'Libro malo', -2020);

-- 4. Insertar un socio con email duplicado (viola UNIQUE)
INSERT INTO socio (nombre, email) VALUES ('Otro Juan', 'juan@email.com');
"""
print("✅ Código de violaciones listo. Pégarlo en DB-Fiddle y verifica los errores.")

---
## Actividad 2: Implementación del modelo de Ventas

Ahora implementaremos el modelo de ventas que diseñaste en el NB1 (o el propuesto).

Modelo lógico de ventas (ejemplo de solución):

```sql
Tablas:
- Cliente (id_cliente, nombre, email, telefono, direccion)
- Categoria (id_categoria, nombre)
- Producto (id_producto, nombre, descripcion, precio, stock, id_categoria)
- Pedido (id_pedido, fecha, estado, id_cliente)
- DetallePedido (id_pedido, id_producto, cantidad, precio_unitario)
```

### 2.1. Crear las tablas

En DB-Fiddle, limpia el panel Schema y pega el siguiente código:

In [ ]:
# Código SQL para el modelo de ventas
sql_ventas = """
CREATE TABLE cliente (
    id_cliente SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    telefono VARCHAR(20),
    direccion VARCHAR(200)
);

CREATE TABLE categoria (
    id_categoria SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL
);

CREATE TABLE producto (
    id_producto SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    descripcion TEXT,
    precio DECIMAL(10,2) NOT NULL CHECK (precio > 0),
    stock INT NOT NULL CHECK (stock >= 0),
    id_categoria INT REFERENCES categoria(id_categoria)
);

CREATE TABLE pedido (
    id_pedido SERIAL PRIMARY KEY,
    fecha DATE NOT NULL DEFAULT CURRENT_DATE,
    estado VARCHAR(20) NOT NULL CHECK (estado IN ('pendiente', 'enviado', 'entregado')),
    id_cliente INT NOT NULL REFERENCES cliente(id_cliente)
);

CREATE TABLE detalle_pedido (
    id_pedido INT REFERENCES pedido(id_pedido),
    id_producto INT REFERENCES producto(id_producto),
    cantidad INT NOT NULL CHECK (cantidad > 0),
    precio_unitario DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (id_pedido, id_producto)
);
"""
print("✅ Código SQL para ventas listo. Pégarlo en DB-Fiddle.")

### 2.2. Insertar datos de prueba y probar integridad

Pega y ejecuta estas inserciones en el panel Query de DB-Fiddle:

In [ ]:
# Inserciones para ventas
inserts_ventas = """
-- Insertar categorías
INSERT INTO categoria (nombre) VALUES ('Electrónica'), ('Ropa'), ('Hogar');

-- Insertar clientes
INSERT INTO cliente (nombre, email, telefono, direccion) VALUES
    ('Carlos López', 'carlos@email.com', '555-1111', 'Calle 1'),
    ('Ana Martínez', 'ana@email.com', '555-2222', 'Calle 2');

-- Insertar productos
INSERT INTO producto (nombre, descripcion, precio, stock, id_categoria) VALUES
    ('Laptop', 'Laptop de alta gama', 1200.00, 10, 1),
    ('Mouse', 'Mouse inalámbrico', 25.50, 50, 1),
    ('Camiseta', 'Camiseta de algodón', 15.00, 100, 2);

-- Insertar pedido
INSERT INTO pedido (fecha, estado, id_cliente) VALUES
    ('2025-03-10', 'pendiente', 1);

-- Insertar detalle del pedido
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES
    (1, 1, 1, 1200.00),
    (1, 2, 2, 25.50);

-- Consultas de verificación
SELECT * FROM cliente;
SELECT * FROM pedido;
SELECT * FROM detalle_pedido;
"""
print("✅ Inserciones para ventas listas. Pégalas en DB-Fiddle.")

### 2.3. Prueba de integridad en ventas

Intenta insertar datos que violen las constraints y observa los errores:

In [ ]:
# Violaciones en ventas
violaciones_ventas = """
-- Insertar producto con precio negativo (viola CHECK)
INSERT INTO producto (nombre, precio, stock) VALUES ('Producto malo', -10, 5);

-- Insertar pedido con estado no válido
INSERT INTO pedido (fecha, estado, id_cliente) VALUES ('2025-03-10', 'inexistente', 1);

-- Insertar detalle de pedido con producto que no existe
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES (1, 999, 1, 10.0);

-- Insertar detalle con cantidad cero (viola CHECK)
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES (1, 1, 0, 10.0);
"""
print("✅ Pruebas de violación listas.")

---
## Actividad 3: Tu propio modelo (ejercicio)

Elige uno de los siguientes enunciados (o el que diseñaste en el NB1) y repite el proceso:

*   **Red Social** (usuarios, publicaciones, comentarios, likes, follows).
*   **Sistema de Reservas de Hotel** (clientes, habitaciones, reservas, servicios).
*   **Gestión de Tareas** (usuarios, proyectos, tareas, etiquetas).

### Instrucciones:

1.  Escribe el modelo lógico (lista de tablas con columnas, PK y FK).
2.  En DB-Fiddle, crea las tablas con sus constraints.
3.  Inserta datos de prueba que respeten las reglas.
4.  Intenta insertar datos que violen las constraints y documenta los errores.

**Comparte tu código SQL en las celdas siguientes (como comentarios o cadenas).**

In [ ]:
# Escribe aquí tu modelo lógico (como comentario)
# Ejemplo para Red Social:
# Tabla usuario (id_usuario, nombre, email, fecha_registro)
# Tabla publicacion (id_publicacion, contenido, fecha, id_usuario)
# Tabla comentario (id_comentario, contenido, fecha, id_usuario, id_publicacion)
# Tabla like_publicacion (id_usuario, id_publicacion)
# Tabla sigue (id_seguidor, id_seguido)

print("Aquí iría tu modelo lógico.")

In [ ]:
# Pega aquí el código SQL de tu modelo (CREATE TABLES) cuando lo tengas listo
mi_sql = """
-- Tu código SQL aquí
"""
print("Listo para copiar a DB-Fiddle.")

In [ ]:
# Pega aquí las inserciones de prueba
mis_inserts = """
-- Tus inserciones aquí
"""
print("Listo.")

---
## Ejercicios para el Estudiante

1.  **Exporta el esquema**: En DB-Fiddle, después de crear las tablas, puedes obtener el script de creación haciendo clic en "Schema" y copiando el texto. Pega ese script en una celda de código para tener un respaldo.

2.  **Añade más constraints**: Por ejemplo, en la biblioteca, añade una constraint `CHECK` para que `fecha_devolucion` sea posterior a `fecha_prestamo`. Pruébala.

3.  **Diseña una consulta**: Escribe una consulta SQL que liste los préstamos actuales (con fecha_devolucion nula) con los nombres de los socios y títulos de los libros.

4.  **Reflexiona**: ¿Por qué es importante definir constraints a nivel de base de datos en lugar de solo en la aplicación?

---
## Conclusiones de la Sesión

*   Hemos implementado modelos lógicos en un motor SQL real (DB-Fiddle), traduciendo entidades y relaciones en tablas con claves primarias y foráneas.
*   Definimos **constraints** (`PRIMARY KEY`, `FOREIGN KEY`, `CHECK`, `UNIQUE`, `NOT NULL`) para garantizar la integridad de los datos.
*   Insertamos datos de prueba y verificamos que las constraints funcionan, observando errores cuando se intentan violar.
*   Esta práctica consolida la conexión entre el modelado conceptual/lógico y la implementación física en SQL.

¡Ahora sabes cómo llevar un diseño de base de datos a la práctica!